In [29]:
import os 
import re
import sys
import json
from pathlib import Path
from typing import Any
import shutil
from collections import defaultdict
from pydantic import ValidationError
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
from PaperReviewTool.src.schema.meta_stru import ExtractionSchema

def scenario_paths(runs_dir: Path, scenario_name: str) -> dict[str, Path]:
    root = runs_dir / scenario_name
    return {
        "root": root,
        "config": root / "config",
        "logs": root / "logs",
        "manifests": root / "manifests",
        "batch": root / "batch",
        "results": root / "results",
        "summary": root / "summary",
    }

def read_json(path: Path) -> Any:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)
    
def write_json(path: Path, data: Any, indent: int = 2) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(data, f, indent=indent, ensure_ascii=False)

def sanitize_filename(name: str) -> str:
    return re.sub(r'[<>:"/\\|?*]', "_", name)

def load_previous_result(results_dir: Path, paper_id: str) -> dict[str, Any] | None:
    path = results_dir / paper_id / "merged_extraction.json"
    if not path.is_file():
        return None
    data = read_json(path)
    return data if isinstance(data, dict) else None

def parse_field_path(loc: list) -> str:
    """Convert Pydantic error location to readable field path"""
    return '.'.join(str(item) for item in loc)

def get_value_from_data(data: dict, loc: list) -> Any:
    """Extract the actual value from the data using the error location"""
    try:
        current = data
        for key in loc:
            if isinstance(current, dict):
                current = current.get(key)
            elif isinstance(current, list):
                try:
                    current = current[int(key)]
                except (ValueError, IndexError, TypeError):
                    return None
            else:
                return None
        return current
    except:
        return None

def truncate_value(value: Any, max_length: int = 200) -> Any:
    """Truncate long values for readability"""
    if isinstance(value, str) and len(value) > max_length:
        return value[:max_length] + f"... (truncated, total length: {len(value)})"
    elif isinstance(value, (list, dict)):
        str_repr = str(value)
        if len(str_repr) > max_length:
            return f"{str_repr[:max_length]}... (truncated, type: {type(value).__name__})"
    return value

def analyze_validation_errors(errors: list[dict], original_data: dict) -> dict:
    """Analyze validation errors and extract field-level information with actual values"""
    field_errors = []
    error_type_counts = defaultdict(int)
    field_error_counts = defaultdict(int)
    invalid_values_by_field = defaultdict(list)
    
    for error in errors:
        field_path = parse_field_path(error['loc'])
        error_type = error['type']
        error_msg = error['msg']
        
        # Get the actual invalid value from the original data
        actual_value = get_value_from_data(original_data, error['loc'])
        truncated_value = truncate_value(actual_value)
        
        # Also get the value from the error input if available
        error_input = error.get('input')
        
        field_error_detail = {
            'field': field_path,
            'type': error_type,
            'message': error_msg,
            'actual_value': actual_value,
            'truncated_value': truncated_value,
            'value_type': type(actual_value).__name__,
            'error_input': error_input,
            'location': error['loc']
        }
        
        # Add context if available
        if 'ctx' in error:
            field_error_detail['context'] = error['ctx']
        
        field_errors.append(field_error_detail)
        
        error_type_counts[error_type] += 1
        field_error_counts[field_path] += 1
        
        # Store invalid values for this field
        invalid_values_by_field[field_path].append({
            'value': truncated_value,
            'full_value': actual_value,
            'type': type(actual_value).__name__,
            'error_type': error_type,
            'error_msg': error_msg
        })
    
    return {
        'field_errors': field_errors,
        'error_type_counts': dict(error_type_counts),
        'field_error_counts': dict(field_error_counts),
        'invalid_values_by_field': dict(invalid_values_by_field)
    }

def normalize_for_schema(data):
    # list → string
    for key in ["pipe_material", "soil_texture"]:
        if data["tile_design"] is not None:
            if isinstance(data.get("tile_design", {}).get(key), list):
                data["tile_design"][key] = "; ".join(data["tile_design"][key])

    # objectives
    if isinstance(data.get("context", {}).get("objectives"), list):
        data["context"]["objectives"] = " ".join(data["context"]["objectives"])

    # numeric → string
    for stat in data.get("results", {}).get("quantitative_stats", []):
        if isinstance(stat.get("value"), (int, float)):
            stat["value"] = str(stat["value"])

    return data


In [32]:

# Configuration
sample_json_filename = Path('samples/sample_meta.json')
# scenario = 's2_gpt5_single'
scenario = 's1_gpt54_single'
# scenario = 's3_gpt5_multi'
run_dir = Path('runs/')

rows = read_json(sample_json_filename)
path = scenario_paths(run_dir, scenario)

# Create validation output directory
validation_dir = path['summary'] / 'validation'
shutil.rmtree(validation_dir)

validation_dir.mkdir(parents=True, exist_ok=True)

# Track validation results
validation_results = {
    'scenario': scenario,
    'total_papers': 0,
    'papers_found': 0,
    'papers_missing': 0,
    'valid_papers': 0,
    'invalid_papers': 0,
    'details': []
}

# Track field-level statistics across all papers
global_field_stats = {
    'total_errors': 0,
    'error_types': defaultdict(int),
    'failed_fields': defaultdict(int),
    'papers_with_field_errors': defaultdict(list),
    'invalid_values_by_field': defaultdict(list)
}

print(f"Starting validation for scenario: {scenario}")
print("=" * 80)

for row in rows:
    validation_results['total_papers'] += 1
    
    doi = str(row.get("doi", "")).strip()
    if not doi:
        validation_results['papers_missing'] += 1
        continue
    
    paper_id = sanitize_filename(doi)
    previous_result = load_previous_result(path['results'], paper_id)
    
    if previous_result is None:
        validation_results['papers_missing'] += 1
        validation_results['details'].append({
            'doi': doi,
            'paper_id': paper_id,
            'status': 'missing',
            'error': 'File not found'
        })
        print(f"❌ MISSING: {doi}")
        continue
    
    validation_results['papers_found'] += 1
    normalized = normalize_for_schema(previous_result)
    try:
        # Validate the result
        validated = ExtractionSchema.model_validate(normalized, strict=False)
        validation_results['valid_papers'] += 1
        
        # Save validated result
        validated_path = validation_dir / paper_id / 'validated.json'
        write_json(validated_path, validated.model_dump())
        
        validation_results['details'].append({
            'doi': doi,
            'paper_id': paper_id,
            'status': 'valid',
            'validated_path': str(validated_path)
        })
        print(f"✅ VALID: {doi}")
        
    except ValidationError as e:
        validation_results['invalid_papers'] += 1
        
        # Parse validation errors with original data
        raw_errors = json.loads(e.json())
        error_analysis = analyze_validation_errors(raw_errors, previous_result)
        
        # Update global statistics
        global_field_stats['total_errors'] += len(raw_errors)
        for error_type, count in error_analysis['error_type_counts'].items():
            global_field_stats['error_types'][error_type] += count
        
        for field_path, count in error_analysis['field_error_counts'].items():
            global_field_stats['failed_fields'][field_path] += count
            global_field_stats['papers_with_field_errors'][field_path].append({
                'doi': doi,
                'paper_id': paper_id
            })
        
        # Collect invalid values globally
        for field_path, invalid_values in error_analysis['invalid_values_by_field'].items():
            for invalid_value_info in invalid_values:
                global_field_stats['invalid_values_by_field'][field_path].append({
                    'paper_id': paper_id,
                    'doi': doi,
                    **invalid_value_info
                })
        
        # Save detailed validation errors
        error_path = validation_dir / paper_id / 'validation_errors.json'
        error_data = {
            'doi': doi,
            'paper_id': paper_id,
            'total_errors': e.error_count(),
            'error_analysis': error_analysis,
            'raw_errors': raw_errors
        }
        write_json(error_path, error_data)
        
        # Save invalid values separately for easy inspection
        invalid_values_path = validation_dir / paper_id / 'invalid_values.json'
        write_json(invalid_values_path, error_analysis['invalid_values_by_field'])
        
        validation_results['details'].append({
            'doi': doi,
            'paper_id': paper_id,
            'status': 'invalid',
            'error_count': e.error_count(),
            'failed_fields': list(error_analysis['field_error_counts'].keys()),
            'error_path': str(error_path),
            'invalid_values_path': str(invalid_values_path)
        })
        
        print(f"❌ INVALID: {doi} ({e.error_count()} errors)")
        print(f"   Failed fields: {', '.join(error_analysis['field_error_counts'].keys())}")

# Convert defaultdicts to regular dicts for JSON serialization
global_field_stats['error_types'] = dict(global_field_stats['error_types'])
global_field_stats['failed_fields'] = dict(global_field_stats['failed_fields'])
global_field_stats['papers_with_field_errors'] = dict(global_field_stats['papers_with_field_errors'])
global_field_stats['invalid_values_by_field'] = dict(global_field_stats['invalid_values_by_field'])

# Save summary report
summary_path = validation_dir / 'validation_summary.json'
write_json(summary_path, validation_results)

# Save field-level statistics with invalid values
field_stats_path = validation_dir / 'field_failure_statistics.json'
write_json(field_stats_path, global_field_stats)

# Create a sorted report of most common field failures
sorted_field_failures = sorted(
    global_field_stats['failed_fields'].items(),
    key=lambda x: x[1],
    reverse=True
)

field_failure_report = {
    'total_unique_fields_with_errors': len(sorted_field_failures),
    'most_common_failures': [
        {
            'field': field,
            'failure_count': count,
            'affected_papers': len(global_field_stats['papers_with_field_errors'][field]),
            'failure_rate': f"{count / validation_results['invalid_papers'] * 100:.1f}%" if validation_results['invalid_papers'] > 0 else "0%",
            'sample_invalid_values': global_field_stats['invalid_values_by_field'][field][:5]  # First 5 examples
        }
        for field, count in sorted_field_failures
    ]
}

field_report_path = validation_dir / 'field_failure_report.json'
write_json(field_report_path, field_failure_report)

# Create a detailed invalid values report
invalid_values_report = {}
for field_path, values in global_field_stats['invalid_values_by_field'].items():
    # Group by error type
    by_error_type = defaultdict(list)
    for val_info in values:
        by_error_type[val_info['error_type']].append(val_info)
    
    invalid_values_report[field_path] = {
        'total_invalid_instances': len(values),
        'unique_error_types': list(by_error_type.keys()),
        'by_error_type': dict(by_error_type),
        'all_values': values
    }

invalid_values_report_path = validation_dir / 'invalid_values_report.json'
write_json(invalid_values_report_path, invalid_values_report)

# Print summary
print("\n" + "=" * 80)
print("VALIDATION SUMMARY")
print("=" * 80)
print(f"Total papers in sample: {validation_results['total_papers']}")
print(f"Papers found: {validation_results['papers_found']}")
print(f"Papers missing: {validation_results['papers_missing']}")
print(f"Valid papers: {validation_results['valid_papers']}")
print(f"Invalid papers: {validation_results['invalid_papers']}")
if validation_results['papers_found'] > 0:
    print(f"\nSuccess rate: {validation_results['valid_papers'] / validation_results['papers_found'] * 100:.1f}%")

print("\n" + "=" * 80)
print("FIELD FAILURE ANALYSIS")
print("=" * 80)
print(f"Total validation errors: {global_field_stats['total_errors']}")
print(f"Unique fields with errors: {len(sorted_field_failures)}")

print("\nTop 10 Most Common Field Failures:")
for i, (field, count) in enumerate(sorted_field_failures[:10], 1):
    affected = len(global_field_stats['papers_with_field_errors'][field])
    print(f"{i:2d}. {field:40s} - {count:3d} errors ({affected} papers)")
    
    # Show a sample invalid value
    if global_field_stats['invalid_values_by_field'][field]:
        sample = global_field_stats['invalid_values_by_field'][field][0]
        print(f"     Sample invalid value: {sample['value']} (type: {sample['type']})")
        print(f"     Error: {sample['error_msg']}")

print("\nError Type Distribution:")
sorted_error_types = sorted(global_field_stats['error_types'].items(), key=lambda x: x[1], reverse=True)
for error_type, count in sorted_error_types[:10]:
    print(f"  {error_type:30s}: {count:3d}")

print(f"\n" + "=" * 80)
print(f"Results saved to: {validation_dir}")
print(f"  - Summary report: {summary_path}")
print(f"  - Field statistics: {field_stats_path}")
print(f"  - Field failure report: {field_report_path}")
print(f"  - Invalid values report: {invalid_values_report_path}")
print(f"\nPer-paper results include:")
print(f"  - validation_errors.json (detailed errors)")
print(f"  - invalid_values.json (actual invalid values)")

Starting validation for scenario: s1_gpt54_single
❌ INVALID: 10.13031/ja.15542 (1 errors)
   Failed fields: methodology.approach_types.0
❌ INVALID: 10.1016/j.jenvman.2023.119220 (1 errors)
   Failed fields: methodology.approach_types.0
❌ INVALID: 10.1016/S0378-3774(00)00136-0 (1 errors)
   Failed fields: methodology.approach_types.1
❌ INVALID: 10.2134/jeq2013.01.0018 (2 errors)
   Failed fields: methodology.approach_types.0, methodology.approach_types.1
❌ INVALID: 10.1016/j.agwat.2022.107597 (3 errors)
   Failed fields: methodology.approach_types.0, methodology.approach_types.1, tile_design.associated_management_practices.0
❌ INVALID: 10.1016/j.agwat.2019.105760 (1 errors)
   Failed fields: methodology.approach_types.0
✅ VALID: 10.1016/j.agwat.2019.03.043
❌ INVALID: 10.1088/1748-9326/acd708 (1 errors)
   Failed fields: methodology.approach_types.0
❌ INVALID: 10.1016/j.agwat.2014.12.003 (1 errors)
   Failed fields: methodology.approach_types.0
❌ INVALID: 10.1016/j.jhydrol.2023.130432 (2

In [24]:
previous_result

{'metadata': {'title': 'Nitrate dynamics in event-driven wetlands',
  'study_type': ['Empirical', 'Modeling'],
  'location': {'country': 'USA',
   'state_province': 'Illinois',
   'county': None,
   'watershed': 'Des Plaines River',
   'coordinates_text': '42°26′N, 87°56′W',
   'spatial_extent': 'Des Plaines River wetland demonstration site near Wadsworth, Illinois; upstream drainage area approximately 520 km^2'}},
 'relevance': {'level': 'Not relevant',
  'is_primary': False,
  'justification': 'The overall objective of the work described here is to develop simple models of nitrate loss for wetlands receiving pulsed loads of non-point source nitrate.'},
 'key_words': ['Treatment wetlands',
  'Event-driven hydrology',
  'Dynamic mass balance modeling',
  'Hydraulic mixing and storage',
  'Tanks-in-series modeling',
  'Residence time distribution',
  'Tracer testing',
  'High-frequency sampling',
  'Hydraulic loading and detention time',
  'Temperature dependence of reaction rates',
  '

In [14]:

rows = read_json(sample_json_filename)
path = scenario_paths(run_dir, scenario)

# Create validation output directory
validation_dir = path['summary'] / 'validation'
validation_dir.mkdir(parents=True, exist_ok=True)

# Track validation results
validation_results = {
    'scenario': scenario,
    'total_papers': 0,
    'papers_found': 0,
    'papers_missing': 0,
    'valid_papers': 0,
    'invalid_papers': 0,
    'details': []
}

# Track field-level statistics across all papers
global_field_stats = {
    'total_errors': 0,
    'error_types': defaultdict(int),
    'failed_fields': defaultdict(int),
    'papers_with_field_errors': defaultdict(list)
}

print(f"Starting validation for scenario: {scenario}")
print("=" * 80)

for row in rows:
    validation_results['total_papers'] += 1
    
    doi = str(row.get("doi", "")).strip()
    if not doi:
        validation_results['papers_missing'] += 1
        continue
    
    paper_id = sanitize_filename(doi)
    previous_result = load_previous_result(path['results'], paper_id)
    
    if previous_result is None:
        validation_results['papers_missing'] += 1
        validation_results['details'].append({
            'doi': doi,
            'paper_id': paper_id,
            'status': 'missing',
            'error': 'File not found'
        })
        print(f"❌ MISSING: {doi}")
        continue
    
    validation_results['papers_found'] += 1
    
    try:
        # Validate the result
        validated = ExtractionSchema.model_validate(previous_result)
        validation_results['valid_papers'] += 1
        
        # Save validated result
        validated_path = validation_dir / paper_id / 'validated.json'
        write_json(validated_path, validated.model_dump())
        
        validation_results['details'].append({
            'doi': doi,
            'paper_id': paper_id,
            'status': 'valid',
            'validated_path': str(validated_path)
        })
        print(f"✅ VALID: {doi}")
        
    except ValidationError as e:
        validation_results['invalid_papers'] += 1
        
        # Parse validation errors
        raw_errors = json.loads(e.json())
        error_analysis = analyze_validation_errors(raw_errors)
        
        # Update global statistics
        global_field_stats['total_errors'] += len(raw_errors)
        for error_type, count in error_analysis['error_type_counts'].items():
            global_field_stats['error_types'][error_type] += count
        
        for field_path, count in error_analysis['field_error_counts'].items():
            global_field_stats['failed_fields'][field_path] += count
            global_field_stats['papers_with_field_errors'][field_path].append({
                'doi': doi,
                'paper_id': paper_id
            })
        
        # Save detailed validation errors
        error_path = validation_dir / paper_id / 'validation_errors.json'
        error_data = {
            'doi': doi,
            'paper_id': paper_id,
            'total_errors': e.error_count(),
            'error_analysis': error_analysis,
            'raw_errors': raw_errors
        }
        write_json(error_path, error_data)
        
        validation_results['details'].append({
            'doi': doi,
            'paper_id': paper_id,
            'status': 'invalid',
            'error_count': e.error_count(),
            'failed_fields': list(error_analysis['field_error_counts'].keys()),
            'error_path': str(error_path)
        })
        
        print(f"❌ INVALID: {doi} ({e.error_count()} errors)")
        print(f"   Failed fields: {', '.join(error_analysis['field_error_counts'].keys())}")

# Convert defaultdicts to regular dicts for JSON serialization
global_field_stats['error_types'] = dict(global_field_stats['error_types'])
global_field_stats['failed_fields'] = dict(global_field_stats['failed_fields'])
global_field_stats['papers_with_field_errors'] = dict(global_field_stats['papers_with_field_errors'])

# Save summary report
summary_path = validation_dir / 'validation_summary.json'
write_json(summary_path, validation_results)

# Save field-level statistics
field_stats_path = validation_dir / 'field_failure_statistics.json'
write_json(field_stats_path, global_field_stats)

# Create a sorted report of most common field failures
sorted_field_failures = sorted(
    global_field_stats['failed_fields'].items(),
    key=lambda x: x[1],
    reverse=True
)

field_failure_report = {
    'total_unique_fields_with_errors': len(sorted_field_failures),
    'most_common_failures': [
        {
            'field': field,
            'failure_count': count,
            'affected_papers': len(global_field_stats['papers_with_field_errors'][field]),
            'failure_rate': f"{count / validation_results['invalid_papers'] * 100:.1f}%" if validation_results['invalid_papers'] > 0 else "0%"
        }
        for field, count in sorted_field_failures
    ]
}

field_report_path = validation_dir / 'field_failure_report.json'
write_json(field_report_path, field_failure_report)

# Print summary
print("\n" + "=" * 80)
print("VALIDATION SUMMARY")
print("=" * 80)
print(f"Total papers in sample: {validation_results['total_papers']}")
print(f"Papers found: {validation_results['papers_found']}")
print(f"Papers missing: {validation_results['papers_missing']}")
print(f"Valid papers: {validation_results['valid_papers']}")
print(f"Invalid papers: {validation_results['invalid_papers']}")
if validation_results['papers_found'] > 0:
    print(f"\nSuccess rate: {validation_results['valid_papers'] / validation_results['papers_found'] * 100:.1f}%")

print("\n" + "=" * 80)
print("FIELD FAILURE ANALYSIS")
print("=" * 80)
print(f"Total validation errors: {global_field_stats['total_errors']}")
print(f"Unique fields with errors: {len(sorted_field_failures)}")

print("\nTop 10 Most Common Field Failures:")
for i, (field, count) in enumerate(sorted_field_failures[:10], 1):
    affected = len(global_field_stats['papers_with_field_errors'][field])
    print(f"{i:2d}. {field:40s} - {count:3d} errors ({affected} papers)")

print("\nError Type Distribution:")
sorted_error_types = sorted(global_field_stats['error_types'].items(), key=lambda x: x[1], reverse=True)
for error_type, count in sorted_error_types[:10]:
    print(f"  {error_type:30s}: {count:3d}")

print(f"\n" + "=" * 80)
print(f"Results saved to: {validation_dir}")
print(f"  - Summary report: {summary_path}")
print(f"  - Field statistics: {field_stats_path}")
print(f"  - Field failure report: {field_report_path}")

Starting validation for scenario: s3_gpt5_multi
✅ VALID: 10.13031/ja.15542


TypeError: analyze_validation_errors() missing 1 required positional argument: 'original_data'

In [10]:
field_failure_report

{'total_unique_fields_with_errors': 12,
 'most_common_failures': [{'field': 'overall_synthesis',
   'failure_count': 1,
   'affected_papers': 1,
   'failure_rate': '14.3%'},
  {'field': 'tile_design.drainage_practices.1',
   'failure_count': 1,
   'affected_papers': 1,
   'failure_rate': '14.3%'},
  {'field': 'tile_design.pipe_material',
   'failure_count': 1,
   'affected_papers': 1,
   'failure_rate': '14.3%'},
  {'field': 'tile_design.soil_texture',
   'failure_count': 1,
   'affected_papers': 1,
   'failure_rate': '14.3%'},
  {'field': 'scale.extraction_FLAG',
   'failure_count': 1,
   'affected_papers': 1,
   'failure_rate': '14.3%'},
  {'field': 'results.quantitative_stats.0.value',
   'failure_count': 1,
   'affected_papers': 1,
   'failure_rate': '14.3%'},
  {'field': 'results.quantitative_stats.1.value',
   'failure_count': 1,
   'affected_papers': 1,
   'failure_rate': '14.3%'},
  {'field': 'results.quantitative_stats.2.value',
   'failure_count': 1,
   'affected_papers': 1,


In [41]:
def scenario_paths(runs_dir: Path, scenario_name: str) -> dict[str, Path]:
    root = runs_dir / scenario_name
    return {
        "root": root,
        "config": root / "config",
        "logs": root / "logs",
        "manifests": root / "manifests",
        "batch": root / "batch",
        "results": root / "results",
        "summary": root / "summary",
    }

def read_json(path: Path) -> Any:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)
    
def sanitize_filename(name: str) -> str:
    return re.sub(r'[<>:"/\\|?*]', "_", name)

def load_previous_result(results_dir: Path, paper_id: str) -> dict[str, Any] | None:
    path = results_dir / paper_id / "merged_extraction.json"
    if not path.is_file():
        # print('Nofile found')
        return None
    data = read_json(path)
    return data if isinstance(data, dict) else None

In [42]:
sample_json_filename = Path('samples/sample_meta.json')
scenario = 's2_gpt5_single'
run_dir = Path('runs/')

In [43]:
rows = read_json(sample_json_filename)
path = scenario_paths(run_dir, scenario)

In [44]:
path

{'root': PosixPath('runs/s2_gpt5_single'),
 'config': PosixPath('runs/s2_gpt5_single/config'),
 'logs': PosixPath('runs/s2_gpt5_single/logs'),
 'manifests': PosixPath('runs/s2_gpt5_single/manifests'),
 'batch': PosixPath('runs/s2_gpt5_single/batch'),
 'results': PosixPath('runs/s2_gpt5_single/results'),
 'summary': PosixPath('runs/s2_gpt5_single/summary')}

In [ ]:
for row in rows:
    doi = str(row.get("doi", "")).strip()
    if not doi:
        continue
    paper_id = sanitize_filename(doi)
    previous_result = load_previous_result(path['results'], paper_id)
    ExtractionSchema.model_validate(previous_result)

In [46]:
paper_id

'10.1029_WR025i007p01717'

ExtractionSchema(metadata=Metadata(title='Influence of Slope on Subsurface Drainage of Hillsides', study_type=['Modeling'], location=None), relevance=Relevance(level='Tile-centered', is_primary=True, justification='drain tubes located $1\\mathrm{m}$ below the soil surface and spaced 30 m apart.'), key_words=['subsurface drainage', 'hillside drainage', 'Richards equation', 'unsaturated and saturated flow', 'numerical modeling', 'finite element method', 'water table dynamics', 'drain flow rates', 'slope effects', 'steady infiltration', 'transient drainage', 'boundary condition effects', 'equipotential analysis'], context=Context(scientific_question='Examine the influence of slope on drain flow rates and water table depths in the center regions of sloping hillsides.', research_gap='Our understanding of the design and performance of subsurface drainage systems on slopes is limited and is based on a small number of field experiments and theoretical analyses.', objectives='To analyze the inf

In [32]:
previous_result